# Build the STEAD subset
This notebook downloads the six STEAD archive chunks sequentially, extracts the metadata and waveform files, keeps the traces required by the WaveDecompNet workflow, and produces `merged.csv` and `merged.hdf5`.

**Important:** The full run is storage-intensive. The source script estimates about **50 GB peak disk usage** and **15–17 GB final output**. Ensure the paths below point to a drive with sufficient free space. The process is resumable through `progress.json`.

## 1. Install dependencies
Run this cell once in the notebook environment. It installs the packages required by the script.

In [1]:
%pip install gdown h5py numpy pandas

   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ------ --------------------------------- 0.5/3.2 MB 1.5 MB/s eta 0:00:02
   --------- ------------------------------ 0.8/3.2 MB 1.7 MB/s eta 0:00:02
   ------------- -------------------------- 1.0/3.2 MB 1.6 MB/s eta 0:00:02
   -------------------------------- ------- 2.6/3.2 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 3.2/3.2 MB 2.9 MB/s eta 0:00:00

   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------- ----------------------------- 1/4 [PySocks]
   -------------------- ------------------- 2/4 [h5py]
   -------------------- ------------------- 2/4 [h5py]
   -------------------- --------------


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and configuration
Edit `WORK` and `OUT` for your machine before running. The defaults below are the Windows paths from the original script.

In [2]:
import os, sys, time, json, zlib, struct, shutil
import numpy as np, pandas as pd, h5py
from numpy.random import default_rng

In [10]:
# ----------------------------- EDIT THESE -----------------------------
WORK = r"C:\Users\HP\OneDrive - iitr.ac.in\Desktop\Program\Python\Data_Dat"      # temporary files (needs ~35 GB free while running)
OUT = r"C:\Users\HP\OneDrive - iitr.ac.in\Desktop\Program\Python\Data_Dat"     # final files (about 17 GB)
N_EVENTS = 100000            # same as the notebook
N2 = 87191                   # POHA noise windows (printed by the notebook)
DRIVE_IDS = {
    1: "1sgLd3eVIrTMcuWtOQRN1fGmiNQ7vc227",
    2: "1ONNNtiOwFodAAeV-ZWp_-gtWhRcYgU_E",
    3: "15bt3WS-mQJmkVY4VOb5LYk45xQBYtkoM",
    4: "12Rw-WI9T21g0hrqtF1KiETYzV4rz177k",
    5: "1rR6bbqu1yE8N_EVIb3kH3wzPPVLFerFX",
    6: "1QfONZf8TBuQlkvKjwJgVMgZub4RDUkH9",
}
RETRY_WAIT_MIN = 30          # wait this long when Drive says "too many users"
MAX_TRIES = 60               # about 30 hours of retrying per chunk

## 3. Keep the machine awake (Windows only)

In [11]:
def keep_awake():
    if os.name == "nt":
        import ctypes
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001)  # no system sleep while running

## 4. Extract selected files from each ZIP archive
The extractor scans the ZIP stream sequentially, as the source script notes that the STEAD ZIP central directory may be broken.

In [12]:
def stream_extract(path, suffix, outdir):
    """Reads the zip sequentially (ignores the broken central directory of the STEAD zips)."""
    SIG, CEN = b"PK\x03\x04", b"PK\x01\x02"
    CH = 1 << 24
    os.makedirs(outdir, exist_ok=True)
    with open(path, "rb") as f:
        pos = 0
        while True:
            p = pos
            while True:
                f.seek(p)
                win = f.read(65536)
                if not win:
                    raise RuntimeError(f"no member ending with {suffix} in {path}")
                j, k = win.find(SIG), win.find(CEN)
                if j >= 0 and (k < 0 or j < k):
                    p += j
                    break
                if k >= 0:
                    raise RuntimeError(f"no member ending with {suffix} in {path}")
                p += max(len(win) - 3, 1)
            f.seek(p)
            h = f.read(30)
            _, ver, flags, method, mt, md, crc, cs, us, fnl, exl = struct.unpack("<4sHHHHHIIIHH", h)
            name = f.read(fnl).decode("utf-8", "replace")
            f.read(exl)
            data_start = p + 30 + fnl + exl
            if flags & 1:
                raise RuntimeError("encrypted member")
            if method == 0 and cs == 0:
                pos = data_start
                continue
            if method != 8:
                raise RuntimeError(f"unsupported compression method {method} for {name}")
            base = os.path.basename(name)
            want = name.endswith(suffix) and "__MACOSX" not in name and not base.startswith("._")
            if want:
                final = os.path.join(outdir, base)
                tmp = final + ".part"
                out = open(tmp, "wb")
                print("  extracting", name, flush=True)
            else:
                out = None
            d = zlib.decompressobj(-15)
            f.seek(data_start)
            consumed = 0
            while not d.eof:
                buf = f.read(CH)
                if not buf:
                    raise RuntimeError(f"archive ended inside member {name}")
                data = d.decompress(buf)
                if out:
                    out.write(data)
                consumed += len(buf)
            pos = data_start + consumed - len(d.unused_data)
            if out:
                out.close()
                os.replace(tmp, final)
                return final

## 5. Download a chunk from Google Drive with retries

In [13]:
def gdown_download(i, dest):
    import gdown
    fid = DRIVE_IDS[i]
    for attempt in range(1, MAX_TRIES + 1):
        try:
            out = gdown.download(id=fid, output=dest + ".part", quiet=False)
            if out and os.path.exists(dest + ".part"):
                os.replace(dest + ".part", dest)
                return
        except Exception as e:
            print(f"chunk{i}: download failed (try {attempt}/{MAX_TRIES}): {repr(e)[:150]}", flush=True)
        if os.path.exists(dest + ".part"):
            os.remove(dest + ".part")
        print(f"  waiting {RETRY_WAIT_MIN} min, then retrying", flush=True)
        time.sleep(RETRY_WAIT_MIN * 60)
    raise RuntimeError(f"chunk{i}: gave up after {MAX_TRIES} tries")

## 6. Compute mean SNR

In [14]:
def mean_snr(values):
    """Same logic as the notebook / repo step2b."""
    out = np.zeros(len(values))
    for i, v in enumerate(values):
        try:
            if np.isnan(v):
                out[i] = v
        except TypeError:
            t = v.replace("[", "").replace("]", "").split()
            out[i] = np.mean([float(x) for x in t])
    return out

## 7. Run the complete pipeline
Chunks are processed one at a time. Completed chunks are recorded in `progress.json` and skipped on subsequent runs.

In [15]:
def run(download_fn=gdown_download):
    keep_awake()
    os.makedirs(WORK, exist_ok=True)
    os.makedirs(os.path.join(OUT, "csv"), exist_ok=True)
    prog_file = os.path.join(OUT, "progress.json")
    done = json.load(open(prog_file))["done"] if os.path.exists(prog_file) else []
    h5_path = os.path.join(OUT, "merged.hdf5")

    for i in range(1, 7):
        csv_keep = os.path.join(OUT, "csv", f"chunk{i}.csv")
        if i in done:
            print(f"chunk{i}: already done, skipping")
            continue
        print(f"=== chunk{i} ===", flush=True)
        arc = os.path.join(WORK, f"chunk{i}.zip")
        cdir = os.path.join(WORK, f"chunk{i}")
        shutil.rmtree(cdir, ignore_errors=True)
        if not os.path.exists(arc):
            download_fn(i, arc)
        csv_i = stream_extract(arc, ".csv", cdir)
        df = pd.read_csv(csv_i, low_memory=False)
        shutil.copy(csv_i, csv_keep)
        print(f"  rows: {len(df)}  {df.trace_category.value_counts().to_dict()}")
        if i == 1:
            dn = df[df.trace_category == "noise"]
            need = set(dn.iloc[default_rng(102).choice(len(dn), N_EVENTS)].trace_name)
        else:
            m = ((df.trace_category == "earthquake_local") & (mean_snr(df.snr_db.to_list()) >= 40)).to_numpy()
            need = set(df[m].trace_name)
        print(f"  traces to keep from this chunk: {len(need)}", flush=True)
        h5_i = stream_extract(arc, ".hdf5", cdir)
        os.remove(arc)
        t0 = time.time()
        with h5py.File(h5_path, "a") as out, h5py.File(h5_i, "r") as src:
            grp = out.require_group("data")
            have = set(grp.keys())
            for n in need - have:
                grp.create_dataset(n, data=np.array(src["data/" + n], dtype=np.float32))
            out.flush()
        print(f"  copied in {time.time() - t0:.0f} s", flush=True)
        shutil.rmtree(cdir, ignore_errors=True)
        done.append(i)
        json.dump({"done": done}, open(prog_file, "w"))

    # ---- merged.csv + final check ----
    parts = []
    for i in range(1, 7):
        d = pd.read_csv(os.path.join(OUT, "csv", f"chunk{i}.csv"), low_memory=False)
        d["chunk"] = i
        parts.append(d)
    df = pd.concat(parts, ignore_index=True)
    df.to_csv(os.path.join(OUT, "merged.csv"), index=False)
    ms = mean_snr(df.snr_db.to_list())
    df_eq = df[(df.trace_category == "earthquake_local") & (ms >= 40)]
    df_noise = df[df.trace_category == "noise"]
    needed = set(df_eq.iloc[default_rng(101).choice(len(df_eq), N_EVENTS)].trace_name)
    needed |= set(df_noise.iloc[default_rng(102).choice(len(df_noise), N_EVENTS)].trace_name)
    needed |= set(df_eq.iloc[default_rng(111).choice(len(df_eq), N2)].trace_name)
    with h5py.File(h5_path, "r") as f:
        keys = set(f["data"].keys())
    print(f"total rows {len(df)} | earthquakes SNR>=40 {len(df_eq)} | noise {len(df_noise)}")
    print(f"needed {len(needed)} | in subset {len(keys)} | missing {len(needed - keys)}")
    assert not (needed - keys), "some needed traces are missing"
    print("DONE. Upload merged.csv and merged.hdf5 from", OUT)

## 8. Execute
Run this cell only after checking the configuration and available disk space. The final validation asserts that all required traces are present.

In [16]:
run()

=== chunk1 ===


Downloading...
From (original): https://drive.google.com/uc?id=1sgLd3eVIrTMcuWtOQRN1fGmiNQ7vc227
From (redirected): https://drive.google.com/uc?id=1sgLd3eVIrTMcuWtOQRN1fGmiNQ7vc227&confirm=t&uuid=ec1187dd-eef8-4264-8ba5-e4bcd81399c5
To: C:\Users\HP\OneDrive - iitr.ac.in\Desktop\Program\Python\Data_Dat\chunk1.zip.part
100%|██████████| 14.6G/14.6G [22:32<00:00, 10.8MB/s]  

  extracting chunk1.csv


  rows: 235426  {'noise': 235426}
  traces to keep from this chunk: 81708
  extracting chunk1.hdf5
  copied in 90 s
=== chunk2 ===
chunk2: download failed (try 1/60): FileURLRetrievalError("Failed to retrieve file url:\n\n\tCannot retrieve the public link of the file. You may need to change\n\tthe permission to 'Any
  waiting 30 min, then retrying


Downloading...
From (original): https://drive.google.com/uc?id=1ONNNtiOwFodAAeV-ZWp_-gtWhRcYgU_E
From (redirected): https://drive.google.com/uc?id=1ONNNtiOwFodAAeV-ZWp_-gtWhRcYgU_E&confirm=t&uuid=6237dea3-8e7b-40da-878b-08efaa1339cd
To: C:\Users\HP\OneDrive - iitr.ac.in\Desktop\Program\Python\Data_Dat\chunk2.zip.part
100%|██████████| 13.7G/13.7G [24:20<00:00, 9.36MB/s]  


  extracting chunk2.csv
  rows: 200000  {'earthquake_local': 200000}
  traces to keep from this chunk: 23972
  extracting chunk2.hdf5
  copied in 27 s
=== chunk3 ===
chunk3: download failed (try 1/60): FileURLRetrievalError("Failed to retrieve file url:\n\n\tToo many users have viewed or downloaded this file recently. Please\n\ttry accessing the file
  waiting 30 min, then retrying
chunk3: download failed (try 2/60): FileURLRetrievalError("Failed to retrieve file url:\n\n\tToo many users have viewed or downloaded this file recently. Please\n\ttry accessing the file
  waiting 30 min, then retrying
chunk3: download failed (try 3/60): FileURLRetrievalError("Failed to retrieve file url:\n\n\tToo many users have viewed or downloaded this file recently. Please\n\ttry accessing the file
  waiting 30 min, then retrying
chunk3: download failed (try 4/60): FileURLRetrievalError("Failed to retrieve file url:\n\n\tToo many users have viewed or downloaded this file recently. Please\n\ttry accessing

... (7 outputs left out)

  rows: 200000  {'earthquake_local': 200000}
  traces to keep from this chunk: 33772
  extracting chunk5.hdf5
  copied in 36 s
=== chunk6 ===


Downloading...
From (original): https://drive.google.com/uc?id=1QfONZf8TBuQlkvKjwJgVMgZub4RDUkH9
From (redirected): https://drive.google.com/uc?id=1QfONZf8TBuQlkvKjwJgVMgZub4RDUkH9&confirm=t&uuid=65f6ab70-a4bd-4462-bbff-0819bca8d531
To: C:\Users\HP\OneDrive - iitr.ac.in\Desktop\Program\Python\Data_Dat\chunk6.zip.part
100%|██████████| 15.7G/15.7G [42:10<00:00, 6.22MB/s]  

  extracting chunk6.csv


  rows: 230231  {'earthquake_local': 230231}
  traces to keep from this chunk: 40280
  extracting chunk6.hdf5
  copied in 43 s
total rows 1265657 | earthquakes SNR>=40 153756 | noise 235426
needed 190038 | in subset 235464 | missing 0
DONE. Upload merged.csv and merged.hdf5 from C:\Users\HP\OneDrive - iitr.ac.in\Desktop\Program\Python\Data_Dat


In [18]:
import h5py, pandas as pd
P = r"C:\Users\HP\OneDrive - iitr.ac.in\Desktop\Program\Python\Data_Dat"

df = pd.read_csv(P + r"\merged.csv", low_memory=False)
print(df.shape, df.trace_category.value_counts().to_dict())

with h5py.File(P + r"\merged.hdf5", "r") as f:
    k = list(f["data"].keys())
    print(len(k), f["data"][k[0]].shape)   # expect (6000, 3)

(1265657, 36) {'earthquake_local': 1030231, 'noise': 235426}
235464 (6000, 3)
